# DINOv2 - improved: fine-tuning + richer features

## What changed from the original notebook

| Original | Improved |
|---|---|
| `dinov2-base` (768-dim) | `dinov2-large` (1024-dim) |
| Frozen extraction only | Two-stage: frozen extraction then fine-tuning of top blocks |
| CLS token only | CLS + mean of patch tokens (2048-dim) |
| 224x224 extraction | 336x336 |

### Why fine-tuning closes the gap with EfficientNetV2S

DINOv2 frozen features are excellent general-purpose representations,
but artist attribution is a specialised task. EfficientNetV2S was fine-tuned
on painting data. Fine-tuning the last few transformer blocks does the same for DINOv2.

### Workflow

1. Extract features at 336px using dinov2-large (CLS + patch mean, 2048-dim)
2. Train a Keras MLP head on frozen features - establishes a baseline
3. Fine-tune the last 4 transformer blocks end-to-end at a very low LR
4. Re-extract updated features from the fine-tuned backbone
5. Re-train the Keras head on the improved features


## 0 - Imports and config

In [1]:
import os, math
import numpy as np
from pathlib import Path
import tensorflow as tf
import keras
from keras import Model, layers
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC
from keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, LearningRateScheduler
from keras.regularizers import l2
import tensorflow_addons as tfa

In [2]:
import torch
import torch.nn as nn
import torch.optim as torch_optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import time

DATA_DIR     = Path('../wikiart_split')
FEATURES_DIR = Path('./dino_features_v2')
CKPTS_DIR    = Path('./Checkpoints')
METRICS_DIR  = Path('./Metrics')

DINO_MODEL    = 'facebook/dinov2-large'
EXTRACT_SIZE  = 336    # used for frozen extraction (Step 1) and re-extraction (Step 4)
FINETUNE_SIZE = 224    # lower resolution for fine-tuning — 2.25x fewer pixels per image,
                       # major speed improvement; the backbone adapts equally well at 224
EXTRACT_BS    = 16
FINETUNE_BS   = 16     # increased from 8 — mixed precision frees enough VRAM to double this
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
N_UNFREEZE    = 4

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  VRAM: {props.total_memory / 1e9:.1f} GB')
FEATURES_DIR.mkdir(exist_ok=True)


Device: cuda
GPU: NVIDIA GeForce GTX 1080  VRAM: 8.6 GB


## Step 1 - Extract richer features: CLS + patch mean

The CLS token summarises the image globally.
Patch tokens carry local spatial information - averaging them gives a
spatial summary that complements the global CLS representation.
Concatenating both (2048-dim for dinov2-large) is richer than CLS alone.


In [3]:
processor = AutoImageProcessor.from_pretrained(
    DINO_MODEL,
    size={'height': EXTRACT_SIZE, 'width': EXTRACT_SIZE},
)
dino = AutoModel.from_pretrained(DINO_MODEL).to(DEVICE)
dino.eval()

n_params = sum(p.numel() for p in dino.parameters())
print(f'Loaded {DINO_MODEL} ({n_params:,} params)')
print(f'Extraction resolution: {EXTRACT_SIZE}x{EXTRACT_SIZE}')


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

Loaded facebook/dinov2-large (304,368,640 params)
Extraction resolution: 336x336


In [4]:
class ArtDataset(Dataset):
    IMG_EXTS = {'.jpg', '.jpeg', '.png'}

    def __init__(self, split_dir, class_names):
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.samples = []
        for class_dir in sorted(split_dir.iterdir()):
            if not class_dir.is_dir(): continue
            idx = self.class_to_idx.get(class_dir.name)
            if idx is None: continue
            for img_path in sorted(class_dir.iterdir()):
                if img_path.suffix.lower() in self.IMG_EXTS:
                    self.samples.append((img_path, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB')
        return img, label, str(path)


def collate_fn(batch):
    images, labels, paths = zip(*batch)
    inputs = processor(images=list(images), return_tensors='pt')
    return inputs, torch.tensor(labels), paths


def extract_features(split, class_names, model=None, tag='v2'):
    feat_path  = FEATURES_DIR / f'{split}_features_{tag}.npy'
    label_path = FEATURES_DIR / f'{split}_labels_{tag}.npy'

    if feat_path.exists() and label_path.exists():
        print(f'  {split} ({tag}): loading from cache')
        return np.load(feat_path), np.load(label_path)

    backbone = model if model is not None else dino
    dataset  = ArtDataset(DATA_DIR / split, class_names)
    loader   = DataLoader(dataset, batch_size=EXTRACT_BS, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)

    all_feats, all_labels = [], []
    print(f'  Extracting {split} ({len(dataset)} images, tag={tag})...')

    backbone.eval()
    with torch.no_grad():
        for i, (inputs, labels, _) in enumerate(loader):
            inputs     = {k: v.to(DEVICE) for k, v in inputs.items()}
            out        = backbone(**inputs)
            cls        = out.last_hidden_state[:, 0, :]
            patch_mean = out.last_hidden_state[:, 1:, :].mean(dim=1)
            feats      = torch.cat([cls, patch_mean], dim=1)
            all_feats.append(feats.cpu().numpy())
            all_labels.append(labels.numpy())
            if (i + 1) % 10 == 0:
                print(f'    {min((i+1)*EXTRACT_BS, len(dataset))}/{len(dataset)}')

    feats  = np.concatenate(all_feats,  axis=0).astype(np.float32)
    labels = np.concatenate(all_labels, axis=0).astype(np.int32)
    np.save(feat_path, feats)
    np.save(label_path, labels)
    print(f'  Saved shape={feats.shape}')
    return feats, labels


In [5]:
_tmp = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train', batch_size=None, image_size=(64, 64))
class_names = _tmp.class_names
N_CLASSES   = len(class_names)
print(f'Classes ({N_CLASSES}): {class_names}')
del _tmp

print('Extracting frozen DINOv2-large features (CLS + patch mean, 336px)...')
train_feats, train_labels = extract_features('train', class_names, tag='frozen')
val_feats,   val_labels   = extract_features('val',   class_names, tag='frozen')
test_feats,  test_labels  = extract_features('test',  class_names, tag='frozen')

FEAT_DIM = train_feats.shape[1]
print(f'Feature dimension: {FEAT_DIM}  (expected 2048 for dinov2-large)')


Found 9326 files belonging to 23 classes.
Classes (23): ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']
Extracting frozen DINOv2-large features (CLS + patch mean, 336px)...
  train (frozen): loading from cache
  val (frozen): loading from cache
  test (frozen): loading from cache
Feature dimension: 2048  (expected 2048 for dinov2-large)


## Step 2 - Train Keras head on frozen features

Establishes a strong baseline before the more expensive fine-tuning step.

In [6]:
KERAS_BATCH = 256
AUTOTUNE    = tf.data.AUTOTUNE

def make_tf_datasets(train_f, train_l, val_f, val_l, test_f, test_l, n_classes):
    oh = lambda l: tf.one_hot(l, n_classes).numpy()
    train_ds = (tf.data.Dataset.from_tensor_slices((train_f, oh(train_l)))
                  .shuffle(len(train_f), seed=123).batch(KERAS_BATCH).prefetch(AUTOTUNE))
    val_ds   = (tf.data.Dataset.from_tensor_slices((val_f, oh(val_l)))
                  .batch(KERAS_BATCH).prefetch(AUTOTUNE))
    test_ds  = (tf.data.Dataset.from_tensor_slices((test_f, oh(test_l)))
                  .batch(KERAS_BATCH).prefetch(AUTOTUNE))
    return train_ds, val_ds, test_ds

train_tf, val_tf, test_tf = make_tf_datasets(
    train_feats, train_labels, val_feats, val_labels, test_feats, test_labels, N_CLASSES)


In [7]:
def build_dino_classifier(input_dim, n_classes=23, dropout=0.4, l2_reg=1e-4, name='dino_classifier'):
    inp = keras.Input(shape=(input_dim,), name='features')
    x   = layers.LayerNormalization(name='input_norm')(inp)
    x   = layers.Dense(1024, kernel_regularizer=l2(l2_reg), name='fc1')(x)
    x   = layers.BatchNormalization(name='bn1')(x)
    x   = layers.Activation('gelu', name='act1')(x)
    x   = layers.Dropout(dropout, name='drop1')(x)
    x   = layers.Dense(512, kernel_regularizer=l2(l2_reg), name='fc2')(x)
    x   = layers.BatchNormalization(name='bn2')(x)
    x   = layers.Activation('gelu', name='act2')(x)
    x   = layers.Dropout(dropout, name='drop2')(x)
    x   = layers.Dense(256, kernel_regularizer=l2(l2_reg), name='fc3')(x)
    x   = layers.BatchNormalization(name='bn3')(x)
    x   = layers.Activation('gelu', name='act3')(x)
    x   = layers.Dropout(dropout / 2, name='drop3')(x)
    out = layers.Dense(n_classes, activation='softmax', dtype='float32', name='head')(x)
    return Model(inp, out, name=name)


def cosine_warmup(base_lr, total_epochs, warmup_epochs=5):
    def sched(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        p = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * p))
    return sched


def train_keras_head(train_ds, val_ds, feat_dim, n_classes,
                     epochs=60, lr=3e-4, ckpt_name='ckpt_dino', log_name='log_dino.csv'):
    model = build_dino_classifier(feat_dim, n_classes)
    model.compile(
        optimizer=tfa.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4),
        loss=CategoricalCrossentropy(label_smoothing=0.1),
        metrics=[CategoricalAccuracy(name='accuracy'),
                 AUC(multi_label=True, name='auc'),
                 tfa.metrics.F1Score(num_classes= n_classes, average='macro', name='f1_score')],
    )
    cbs = [
        ModelCheckpoint(f'{ckpt_name}.keras', monitor='val_loss', save_best_only=True, verbose=1),
        CSVLogger(log_name),
        LearningRateScheduler(cosine_warmup(lr, epochs, warmup_epochs=5)),
        EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1),
    ]
    model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=cbs, verbose=1)
    return model


print('Training Keras head on frozen DINOv2-large features...')
model_dino_frozen = train_keras_head(
    train_tf, val_tf, FEAT_DIM, N_CLASSES,
    ckpt_name=CKPTS_DIR / 'ckpt_dino_frozen', log_name= METRICS_DIR / 'log_dino_frozen.csv'
)

frozen_results = model_dino_frozen.evaluate(test_tf, return_dict=True, verbose=0)
print('Frozen backbone - test results:')
for k, v in frozen_results.items():
    print(f'  {k}: {v:.4f}')


Training Keras head on frozen DINOv2-large features...
Epoch 1/60
36/37 [============================>.] - ETA: 0s - loss: 3.4072 - accuracy: 0.1238 - auc: 0.6257 - f1_score: 0.1025
Epoch 1: val_loss improved from inf to 2.77438, saving model to Checkpoints\ckpt_dino_frozen.keras
37/37 [==============================] - 5s 48ms/step - loss: 3.4053 - accuracy: 0.1246 - auc: 0.6265 - f1_score: 0.1030 - val_loss: 2.7744 - val_accuracy: 0.3614 - val_auc: 0.8286 - val_f1_score: 0.2793 - lr: 6.0000e-05
Epoch 2/60
36/37 [============================>.] - ETA: 0s - loss: 2.6762 - accuracy: 0.3535 - auc: 0.8215 - f1_score: 0.2936
Epoch 2: val_loss improved from 2.77438 to 2.25475, saving model to Checkpoints\ckpt_dino_frozen.keras
37/37 [==============================] - 1s 29ms/step - loss: 2.6741 - accuracy: 0.3548 - auc: 0.8221 - f1_score: 0.2953 - val_loss: 2.2548 - val_accuracy: 0.5110 - val_auc: 0.9106 - val_f1_score: 0.4316 - lr: 1.2000e-04
Epoch 3/60
35/37 [===========================>.

## Step 3 - Fine-tune dinov2-base (feasible on GTX 1080)

**Why not dinov2-large?**

Backpropagating through dinov2-large (307M params) at batch=16 requires ~9-10 GB
of activation memory — more than the 1080's 8 GB VRAM. The model silently spills
to system RAM, making each batch take ~858 seconds instead of seconds.

**The switch: dinov2-base for fine-tuning only**

- `dinov2-base`: 86M params, 768-dim hidden states, 12 transformer blocks
- Fits comfortably in 8 GB VRAM; each batch takes ~1-3 seconds
- We unfreeze the last 2 blocks (vs 4 for large) — proportionally equivalent
- Feature dim becomes 2 * 768 = 1536 (CLS + patch mean)

The frozen dinov2-large results (Steps 1-2) are kept as-is. The fine-tuned
dinov2-base adds a second data point: a lighter model that has actually adapted
its representations to painting data. Both are reported in the final comparison.


In [8]:
# Load dinov2-base specifically for fine-tuning.
# This is a SEPARATE model from `dino` (dinov2-large used for frozen extraction).
# It is smaller, fits in VRAM, and will have its top blocks fine-tuned.
DINO_BASE_MODEL  = 'facebook/dinov2-base'
FINETUNE_SIZE    = 224

finetune_processor = AutoImageProcessor.from_pretrained(
    DINO_BASE_MODEL,
    size={'height': FINETUNE_SIZE, 'width': FINETUNE_SIZE},
)
dino_base = AutoModel.from_pretrained(DINO_BASE_MODEL).to(DEVICE)
dino_base.eval()

base_params = sum(p.numel() for p in dino_base.parameters())
print(f'Loaded {DINO_BASE_MODEL} ({base_params:,} params)')
print(f'Fine-tuning resolution: {FINETUNE_SIZE}x{FINETUNE_SIZE}')


class ArtDatasetFull(Dataset):
    IMG_EXTS = {'.jpg', '.jpeg', '.png'}

    def __init__(self, split_dir, class_names):
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.samples = []
        for class_dir in sorted(split_dir.iterdir()):
            if not class_dir.is_dir(): continue
            idx = self.class_to_idx.get(class_dir.name)
            if idx is None: continue
            for img_path in sorted(class_dir.iterdir()):
                if img_path.suffix.lower() in self.IMG_EXTS:
                    self.samples.append((img_path, idx))

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        return Image.open(path).convert('RGB'), label


def collate_full(batch):
    images, labels = zip(*batch)
    inputs = finetune_processor(images=list(images), return_tensors='pt')
    return inputs, torch.tensor(labels, dtype=torch.long)


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded facebook/dinov2-base (86,580,480 params)
Fine-tuning resolution: 224x224


In [9]:
class DinoWithHead(nn.Module):
    """DINOv2 backbone + linear head for end-to-end fine-tuning.
    Only used during the fine-tuning phase — afterwards the backbone
    is detached and used for feature re-extraction."""
    def __init__(self, backbone, feat_dim, n_classes):
        super().__init__()
        self.backbone = backbone
        self.head     = nn.Linear(feat_dim, n_classes)

    def forward(self, inputs):
        out        = self.backbone(**inputs)
        cls        = out.last_hidden_state[:, 0, :]
        patch_mean = out.last_hidden_state[:, 1:, :].mean(dim=1)
        return self.head(torch.cat([cls, patch_mean], dim=1))

In [10]:
def unfreeze_top_blocks(model, n_blocks):
    for p in model.parameters():
        p.requires_grad = False
    total_blocks = len(model.encoder.layer)
    for block in model.encoder.layer[total_blocks - n_blocks:]:
        for p in block.parameters():
            p.requires_grad = True
    if hasattr(model, 'layernorm'):
        for p in model.layernorm.parameters():
            p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Unfroze last {n_blocks}/{total_blocks} blocks: {trainable:,}/{total:,} params trainable')


def finetune_backbone(backbone, class_names, n_unfreeze=N_UNFREEZE,
                      epochs=10, lr=5e-6, warmup_epochs=2):
    import copy
    backbone   = copy.deepcopy(backbone)
    unfreeze_top_blocks(backbone, n_unfreeze)
    feat_dim   = 2 * backbone.config.hidden_size
    full_model = DinoWithHead(backbone, feat_dim, len(class_names)).to(DEVICE)

    train_loader = DataLoader(ArtDatasetFull(DATA_DIR / 'train', class_names),
                              batch_size=FINETUNE_BS, shuffle=True,
                              collate_fn=collate_full, num_workers=0)
    val_loader   = DataLoader(ArtDatasetFull(DATA_DIR / 'val', class_names),
                              batch_size=FINETUNE_BS, shuffle=False,
                              collate_fn=collate_full, num_workers=0)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch_optim.AdamW(
        filter(lambda p: p.requires_grad, full_model.parameters()),
        lr=lr, weight_decay=1e-5)
    warmup_steps = warmup_epochs * len(train_loader)
    scheduler    = torch_optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: min(1.0, step / max(1, warmup_steps)))

    # Mixed precision: halves activation memory and memory bandwidth usage.
    # On GTX 1080 (no Tensor Cores) this still gives ~1.5-2x speedup from
    # reduced memory pressure. GradScaler prevents float16 underflow on gradients.
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))

    best_val_loss = float('inf')
    best_state    = None
    n_batches     = len(train_loader)

    # Print a realistic time estimate before the first epoch starts
    print(f'Fine-tuning: {epochs} epochs x {n_batches} batches/epoch')
    print(f'Images: {FINETUNE_SIZE}x{FINETUNE_SIZE}, batch={FINETUNE_BS}, mixed_precision=True')
    print('Running a timing batch to estimate total time...')

    full_model.train()
    t0 = time.time()
    sample_inputs, sample_labels = next(iter(train_loader))
    sample_inputs = {k: v.to(DEVICE) for k, v in sample_inputs.items()}
    sample_labels = sample_labels.to(DEVICE)
    with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
        scaler.scale(criterion(full_model(sample_inputs), sample_labels)).backward()
    scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
    secs_per_batch = time.time() - t0
    est_hours = secs_per_batch * n_batches * epochs / 3600
    print(f'~{secs_per_batch:.1f}s/batch -> estimated total: {est_hours:.1f} hours\n')

    for epoch in range(epochs):
        full_model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0
        epoch_start = time.time()

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            labels = labels.to(DEVICE)
            optimizer.zero_grad()

            # autocast: runs the forward pass in float16 where safe,
            # keeping numerically sensitive ops in float32 automatically
            with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
                logits = full_model(inputs)
                loss   = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(full_model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            t_loss    += loss.item() * labels.size(0)
            t_correct += (logits.detach().argmax(1) == labels).sum().item()
            t_total   += labels.size(0)

            # Progress within epoch: print every 50 batches
            if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == n_batches:
                elapsed = time.time() - epoch_start
                pct     = (batch_idx + 1) / n_batches
                eta     = elapsed / pct * (1 - pct)
                print(f'  [{epoch+1}/{epochs}] batch {batch_idx+1}/{n_batches}  '
                      f'loss={t_loss/t_total:.4f}  '
                      f'elapsed={elapsed:.0f}s  eta={eta:.0f}s', end='\r')

        print()  # newline after the \r progress line

        full_model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
                labels = labels.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
                    logits = full_model(inputs)
                v_loss    += criterion(logits, labels).item() * labels.size(0)
                v_correct += (logits.argmax(1) == labels).sum().item()
                v_total   += labels.size(0)

        tl = t_loss / t_total
        vl = v_loss / v_total
        epoch_mins = (time.time() - epoch_start) / 60
        print(f'Epoch {epoch+1:02d}/{epochs}  '
              f'train_loss={tl:.4f}  train_acc={t_correct/t_total:.4f}  '
              f'val_loss={vl:.4f}  val_acc={v_correct/v_total:.4f}  '
              f'({epoch_mins:.1f} min)')

        if vl < best_val_loss:
            best_val_loss = vl
            best_state    = {k: v.cpu().clone()
                             for k, v in full_model.backbone.state_dict().items()}
            print(f'  New best val_loss: {best_val_loss:.4f}')

    full_model.backbone.load_state_dict(best_state)
    print(f'Fine-tuning complete. Best val_loss: {best_val_loss:.4f}')
    return full_model.backbone


In [ ]:
# Fine-tune dinov2-base with 2 blocks unfrozen.
# 2/12 blocks = proportionally similar to 4/24 in dinov2-large.
N_UNFREEZE_BASE = 2

print(f'Fine-tuning {DINO_BASE_MODEL}: unfreezing last {N_UNFREEZE_BASE} transformer blocks...')
dino_finetuned = finetune_backbone(
    dino_base,          # base model — fits in VRAM
    class_names,
    n_unfreeze=N_UNFREEZE_BASE,
    epochs=10,
    lr=5e-6,
    warmup_epochs=2,
)


Fine-tuning facebook/dinov2-base: unfreezing last 2 transformer blocks...
Unfroze last 2/12 blocks: 14,180,352/86,580,480 params trainable


C:\Users\franc\AppData\Local\Temp\ipykernel_9172\1965001710.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))


Fine-tuning: 10 epochs x 583 batches/epoch
Images: 224x224, batch=16, mixed_precision=True
Running a timing batch to estimate total time...


C:\Users\franc\AppData\Local\Temp\ipykernel_9172\1965001710.py:58: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):


## Step 4 - Re-extract features and retrain Keras head

In [ ]:
# Re-extract features from the fine-tuned dinov2-base backbone.
# Note: extraction still uses the 336px processor (the original `processor` variable)
# to maintain consistent resolution with the frozen dinov2-large features.
# The fine-tuned backbone handles variable resolution via position embedding interpolation.
print('Re-extracting features from fine-tuned dinov2-base backbone (at 336px)...')
train_feats_ft, train_labels_ft = extract_features('train', class_names, model=dino_finetuned, tag='base_finetuned')
val_feats_ft,   val_labels_ft   = extract_features('val',   class_names, model=dino_finetuned, tag='base_finetuned')
test_feats_ft,  test_labels_ft  = extract_features('test',  class_names, model=dino_finetuned, tag='base_finetuned')

FEAT_DIM_FT = train_feats_ft.shape[1]
print(f'Fine-tuned feature dimension: {FEAT_DIM_FT}  (expected 1536 for dinov2-base)')

train_tf_ft, val_tf_ft, test_tf_ft = make_tf_datasets(
    train_feats_ft, train_labels_ft,
    val_feats_ft,   val_labels_ft,
    test_feats_ft,  test_labels_ft,
    N_CLASSES
)


In [ ]:
print('Training Keras head on fine-tuned dinov2-base features...')
model_dino_ft = train_keras_head(
    train_tf_ft, val_tf_ft,
    FEAT_DIM_FT,   # 1536 for dinov2-base (was 2048 for dinov2-large)
    N_CLASSES,
    ckpt_name=CKPTS_DIR / 'ckpt_dino_base_finetuned',
    log_name=METRICS_DIR / 'log_dino_base_finetuned.csv'
)

ft_results = model_dino_ft.evaluate(test_tf_ft, return_dict=True, verbose=0)
print('Fine-tuned dinov2-base - test results:')
for k, v in ft_results.items():
    print(f'  {k}: {v:.4f}')

print('=' * 60)
print('Comparison: frozen dinov2-large vs fine-tuned dinov2-base')
print('=' * 60)
print(f"{'Metric':<15} {'Frozen-large':>14} {'FT-base':>10} {'Delta':>8}")
print('-' * 52)
for k in frozen_results:
    if k not in ft_results:
        continue
    delta = ft_results[k] - frozen_results[k]
    sign  = '+' if delta >= 0 else ''
    print(f'{k:<15} {frozen_results[k]:>14.4f} {ft_results[k]:>10.4f} {sign}{delta:>7.4f}')
print()
print('Note: frozen uses dinov2-large (2048-dim), fine-tuned uses dinov2-base (1536-dim).')
print('The fine-tuned model has domain-adapted representations; the frozen model has')
print('richer general-purpose features. Both comparisons are valid and interesting.')
